In [1]:
import torchvision.transforms as transforms

mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    #transforms.Normalize((0.5,), (1.0,))
])

from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
import requests

download_root = './data/mnist'
train_dataset = MNIST(download_root, transform=mnist_transform, train=True, download=True)
test_dataset = MNIST(download_root, transform=mnist_transform, train=False, download=True)
batch_size = 200
dataset_train = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [00:01<00:00, 5.08MB/s]


Extracting ./data/mnist/MNIST/raw/train-images-idx3-ubyte.gz to ./data/mnist/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 65.4kB/s]


Extracting ./data/mnist/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/mnist/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1.65M/1.65M [00:06<00:00, 245kB/s]


Extracting ./data/mnist/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/mnist/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4.54k/4.54k [00:00<00:00, 6.53MB/s]

Extracting ./data/mnist/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/mnist/MNIST/raw



In [2]:
import torch.nn as nn

class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.layer3 = nn.Sequential(
            nn.Linear(in_features=64*6*6, out_features=600),
            nn.Linear(600, 120),
            nn.Linear(120, 10)
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = x.view(x.size(0), -1)
        x = self.layer3(x)
        return x

In [3]:
model = Model()

from torch.optim import optimizer
import torch
import numpy as np

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
#scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer=optimizer, lr_lambda=lambda epoch: 0.99 ** epoch)

# train
n_epoch = 10
for epoch in range(1, n_epoch+1):
    train_losses = []
    train_acc = []
    for idx, (x, y) in enumerate(dataset_train):
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        train_losses.append(loss.item())
        train_acc.append( (torch.max(pred, 1)[1] == y).sum()/batch_size )
        loss.backward()
        optimizer.step()
        #scheduler.step()
    print('epoch: {}, loss:{}, acc:{}'.format(epoch, sum(train_losses)/(idx+1), sum(train_acc)/(idx+1) ))

epoch: 1, loss:4.396769611400862, acc:0.8810839056968689
epoch: 2, loss:0.0833455239298443, acc:0.9754170775413513
epoch: 3, loss:0.05863791467932363, acc:0.9822335839271545
epoch: 4, loss:0.04706363997111718, acc:0.9859495759010315
epoch: 5, loss:0.04094929684263964, acc:0.9873835444450378
epoch: 6, loss:0.03500422207561011, acc:0.9892662763595581
epoch: 7, loss:0.03254496867002066, acc:0.9893161058425903
epoch: 8, loss:0.0322984651295701, acc:0.9891331195831299
epoch: 9, loss:0.026255883749496813, acc:0.9915326833724976
epoch: 10, loss:0.025108102209633217, acc:0.992165744304657


In [ ]:
dataset_test = DataLoader(test_dataset, batch_size=batch_size)
test_acc = []
for i, (x, y) in enumerate(dataset_test):
    test_pred = model(x)
    test_acc.append((torch.max(test_pred, 1)[1] == y).sum()/batch_size)
print('test acc:{}'.format(sum(test_acc)/(len(test_acc) )) )

test acc:0.9616999626159668
